# Colab runner

Runs the `make` targets on a Colab GPU, following SPEC section 16. This
notebook holds no project logic. Never edit code in the Colab session.
Edit on the laptop, commit, push, and check out the new commit here.

Connect from VS Code with the Colab extension (Select Kernel, Colab, New
Colab Server) and pick a T4 runtime (D-013).

## 1. Check out the commit to run

Reported runs use a clean, tagged commit. Development runs can use a branch.

In [ ]:
REPO_URL = "https://github.com/nickmelamed/retinal-vessel-segmentation.git"
REF = "v0.1.0"  # the tag (or branch, for development runs) to run
VARIANT = "baseline"  # configs/<VARIANT>.yaml

In [ ]:
!git clone --quiet {REPO_URL} /content/repo
%cd /content/repo
!git checkout --quiet {REF}
!git log --oneline -1
!git status --short

## 2. Install the locked environment

The same `uv sync --locked` that CI runs (D-011). The environment check
should show `COLAB_RELEASE_TAG` set, since runs record `compute_platform`
from it, and a Tesla T4.

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh

In [ ]:
import os

# make calls uv by name, so its install directory must be on PATH.
os.environ["PATH"] = f"{os.environ['HOME']}/.local/bin:{os.environ['PATH']}"

In [ ]:
!uv sync --locked

In [ ]:
!uv run python --version
!uv run python -c "import tensorflow as tf; print(tf.__version__)"
!uv run python -c "import tensorflow as tf; print(tf.config.list_physical_devices('GPU'))"
!nvidia-smi
!echo "COLAB_RELEASE_TAG=${COLAB_RELEASE_TAG:-not set}"

## 3. Upload and check the data

Upload the DRIVE folder (with `training/` and `test/` inside) through the
extension, then point `DRIVE_DIR` at it. The checksums must match the
committed `data/CHECKSUMS.sha256` before any training.

To resume an interrupted run on a new machine, also upload the last
`run_outputs.zip` to `/content` and restore it with
`!unzip -oq /content/run_outputs.zip -d /content/repo`. It holds the
database, the manifests, the saved predictions, and the checkpoints, and
phase 3 needs the predictions of every fold.

In [ ]:
%env DRIVE_DIR=/content/DRIVE
!make check-data

## 4. Train

Rerunning this cell after a disconnect resumes the same run. Complete folds
are skipped and an unfinished one is retrained.

In [ ]:
!make train VARIANT={VARIANT}

## 5. Download the outputs

Run this after each fold if the session might end, then download
`/content/run_outputs.zip` through the extension. On the laptop, unzip it
into the repo and run `make verify-checkpoints` to check the checkpoint
hashes against the database. It fails on a run that is still training, so
check a mid-run download with `make verify-checkpoints UNFINISHED=1`. Once
the run is finished, `make evaluate` computes the remaining metrics on the
laptop.

In [ ]:
!zip -qr /content/run_outputs.zip results models
!ls -lh /content/run_outputs.zip